# Implementing a simple parallel workflow consisiting 3 parallel nodes performing 3 individual tasks independently


In [14]:
from typing import TypedDict, Annotated
from langchain_groq import ChatGroq
from langgraph.graph import START, END, StateGraph
from operator import add

In [ ]:
class WorkflowState(TypedDict):
    query: str
    summary: Annotated[str, ""]
    key_points: Annotated[str, ""]
    action_items: Annotated[str, ""]
    final_output: Annotated[str, ""]

In [3]:
# model
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.4)


In [15]:
def input_node(state: WorkflowState):
    return state

def summarizer(state: WorkflowState):
    prompt = f"Summarize the following query: {state['query']}"
    summary = model.invoke(prompt)
    return {"summary": summary.content if hasattr(summary, 'content') else str(summary)}

def key_points_extractor(state: WorkflowState):
    prompt = f"Extract key points from the following summary: {state['query']}"
    key_points = model.invoke(prompt)
    return {"key_points": key_points.content if hasattr(key_points, 'content') else str(key_points)}

def action_items_generator(state: WorkflowState):
    prompt = f"Generate action items based on the following key points: {state['query']}"
    action_items = model.invoke(prompt)
    return {"action_items": action_items.content if hasattr(action_items, 'content') else str(action_items)}

def aggregator(state: WorkflowState):
    final_output = f"Summary: {state['summary']}\nKey Points: {state['key_points']}\nAction Items: {state['action_items']}"
    return {"final_output": final_output}

In [16]:
# Initalizing the graph:
graph = StateGraph(WorkflowState)

# Adding Nodes:
graph.add_node("Input_Node",input_node)
graph.add_node("summarizer", summarizer)
graph.add_node("key_points_extractor", key_points_extractor)
graph.add_node("action_items_generator", action_items_generator)
graph.add_node("aggregator", aggregator)

# Edges Joining
graph.add_edge(START, "Input_Node")
graph.add_edge("Input_Node", "summarizer")
graph.add_edge("Input_Node", "key_points_extractor")
graph.add_edge("Input_Node", "action_items_generator")
graph.add_edge("summarizer", "aggregator")
graph.add_edge("key_points_extractor", "aggregator")
graph.add_edge("action_items_generator", "aggregator")
graph.add_edge("aggregator", END)

workflow = graph.compile()

In [17]:
# Executing the graph:
initial_state = WorkflowState(query="What are the key takeaways from the latest market trends?")
final_state = workflow.invoke(initial_state)

print(final_state['final_output'])

Summary: The key takeaways from the latest market trends typically include:

1. **Shifts in consumer behavior**: Changes in how people shop, interact with brands, and make purchasing decisions.
2. **Technological advancements**: Emerging technologies, such as AI, blockchain, and the Internet of Things (IoT), and their impact on industries.
3. **Economic indicators**: Insights into economic growth, inflation, interest rates, and their effects on markets.
4. **Industry disruptions**: New entrants, innovations, and business models that are disrupting traditional industries.
5. **Sustainability and social responsibility**: Growing importance of environmental, social, and governance (ESG) factors in business and investment decisions.
6. **Global market trends**: Trends and opportunities in emerging markets, trade policies, and geopolitical shifts.

These takeaways can help businesses, investors, and individuals stay informed and make informed decisions in a rapidly changing market landscape